# MiniMax H3 on ComfyUI — Colab backend

Runs the open-weights **MiniMax H3** video model on a Colab GPU and exposes ComfyUI
through a cloudflared tunnel so your local machine can submit workflows via the API.

**Requirements**
- GPU runtime: *Runtime > Change runtime type > GPU*. A100/L4 recommended;
  T4 (16 GB) works via offloading but is very slow.
- ~45 GB free disk for the weights below.

**Weights used (Comfy-Org/MiniMax-H3, ~42 GB total)**
- `diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors` (21 GB) — T2V + first/last-frame I2V
- `text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors` (16 GB)
- `vae/minimax_h3_video_vae_int8_convrot.safetensors` + `vae/minimax_h3_audio_vae_fp32.safetensors`
- `loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors` (optional turbo mode)

> Commercial use of locally generated outputs requires a MiniMax commercial license
> (sold via Comfy). Personal/experimental use is fine.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Enable a GPU runtime first: Runtime > Change runtime type > GPU" 

## 1. Install ComfyUI (>= 0.30.0)

In [ ]:
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
%cd /content/ComfyUI
!pip -q install -r requirements.txt
!pip -q install hf_transfer
!python -c "import comfyui_version; print('ComfyUI', comfyui_version.__version__)" 

## 2. Download model weights (~42 GB)

`DIFFUSION_FILE`: `int8_convrot` is preferred with PyTorch built for CUDA 13.0+.
If you hit quant/kernel errors, switch to `minimax_h3_fl2va_pruned_fp8_scaled.safetensors`.

In [ ]:
%env HF_HUB_ENABLE_HF_TRANSFER=1
import os
from huggingface_hub import hf_hub_download

# Which workflows do you need? t2v/i2v share the fl2va weights; r2v adds ~23 GB.
MODES = ["t2v", "i2v"]          # add "r2v" for reference-to-video
DIFFUSION_FILE = "minimax_h3_fl2va_pruned_int8_convrot.safetensors"  # or ..._fp8_scaled.safetensors

REPO = "Comfy-Org/MiniMax-H3"
FILES = [
    "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    "vae/minimax_h3_video_vae_int8_convrot.safetensors",
    "vae/minimax_h3_audio_vae_fp32.safetensors",
]
if any(m in MODES for m in ("t2v", "i2v")):
    FILES += [
        f"diffusion_models/{DIFFUSION_FILE}",
        "loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors",
    ]
if "r2v" in MODES:
    FILES += [
        "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
        "loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",
    ]

for rel in FILES:
    dst = f"/content/ComfyUI/models/{rel}"
    if os.path.exists(dst) or os.path.islink(dst):
        print("exists:", rel); continue
    src = hf_hub_download(REPO, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    os.symlink(src, dst)
    print("linked:", rel)

## 3. Launch ComfyUI + cloudflared tunnel

Run this cell; it prints a public `https://*.trycloudflare.com` URL.
Copy it into your local `.env` as `COMFYUI_URL`.

In [ ]:
import subprocess, re, time, threading, urllib.request

subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared", shell=True, check=True)

comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=open("/content/comfyui.log", "w"), stderr=subprocess.STDOUT,
)
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

public_url = None
def read_tunnel():
    global public_url
    for line in tunnel.stdout:
        m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
        if m and public_url is None:
            public_url = m.group(0)
            print("tunnel URL:", public_url, flush=True)

threading.Thread(target=read_tunnel, daemon=True).start()

for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        break
    except Exception:
        time.sleep(2)

while public_url is None:
    time.sleep(1)

print("\n=== READY ===")
print("COMFYUI_URL =", public_url)
print("Open it in a browser for the ComfyUI UI, or use it from the local client.")

## 4. Keep alive / watch logs

Keep this cell running while you generate. Colab still disconnects idle runtimes
(~90 min free tier), so don't leave it unattended.

In [ ]:
!tail -f /content/comfyui.log